In [1]:
import os
import sys
import json
import math
import hashlib
import shutil
import time
import random
import inspect
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
from kaggle_secrets import UserSecretsClient

# --- KAGGLE SECRETS & PATH SETUP ---
try:
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kaggle_api")
except Exception as e:
    print(f"[WARNING] Kaggle secret initialization bypassed: {e}")

WORK = Path("/kaggle/working")
PARQUET = WORK / "submission.parquet"
COMP = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
ENV_DIR = COMP / "environment_files"
WHEELS = COMP / "arc_agi_3_wheels"
WORK.mkdir(exist_ok=True)

# Immediate Bootstrap Protection Layer
pd.DataFrame({
    "row_id": ["boot_0"],
    "game_id": ["boot"],
    "end_of_game": [True],
    "score": [1]}).to_parquet(str(PARQUET), index=False)

# Offline Wheel Installation
os.system(f"pip install --no-index --find-links {WHEELS} arc-agi -q")

try:
    import arcengine
    from arcengine import FrameData, GameAction, GameState
except ImportError:
    print("[ERROR] Failed to bind local engine packages.")
    sys.exit(1)

# --- GLOBAL MEMORY ARCHITECTURE (DEQGT V19) ---
_WIN_SEQS = {}     
_LOSS_TAB = {}     
_TRANS_TAB = {}    

class DEQGTCrystalAgent:
    MAX_ACTIONS = 100
    
    def __init__(self, game_id):
        self.game_id = game_id
        self._step = 0
        self.K_gain = 0.5  
        self.last_sig = None
        self.sig_count = 0
        self.undo_count = 0

    def _get_sig(self, grid):
        h = 17
        if grid.size == 0:
            return h
        flat = grid.flatten()
        for i, px in enumerate(flat):
            h = (h * 1000003 + int(px) * (i + 1)) % 1000000007
        return h

    def _get_grid(self, fd):
        raw = fd.frame[-1] if (hasattr(fd, 'frame') and fd.frame) else []
        return np.array(raw, dtype=np.int32) if raw else np.zeros((1, 1), dtype=np.int32)

    def _simulate_action(self, cfd, act_name):
        grid = self._get_grid(cfd).copy()
        if act_name == "ACTION1":
            grid = np.roll(grid, -1, axis=0)
        elif act_name == "ACTION2":
            grid = np.roll(grid, 1, axis=0)
        elif act_name == "ACTION3":
            grid = np.roll(grid, -1, axis=1)
        elif act_name == "ACTION4":
            grid = np.roll(grid, 1, axis=1)
        elif act_name == "ACTION5":
            grid = np.fliplr(grid)
        elif act_name == "ACTION7":
            grid = np.rot90(grid)
        elif act_name == "ACTION6" and grid.size > 0:
            grid = (grid + 1) % 10
        return arcengine.FrameData(
            grid=grid.tolist(),
            state=cfd.state,
            levels_completed=cfd.levels_completed,
            available_actions=cfd.available_actions
        )

    def _calculate_invariants(self, grid):
        if grid.size == 0:
            return np.array([1.0])
        row_sums = np.sum(grid, axis=1) + 1e-5
        col_sums = np.sum(grid, axis=0) + 1e-5
        inv_row = row_sums[0] / row_sums
        inv_col = col_sums[0] / col_sums
        return np.concatenate([inv_row, inv_col])

    def _evaluate_feedback_loop(self, fd, seq):
        curr = fd
        start_grid = self._get_grid(fd)
        target_symmetrical = np.fliplr(start_grid)
        for a in seq:
            curr = self._simulate_action(curr, a)
        g_final = self._get_grid(curr)
        error_field = target_symmetrical - g_final
        error_norm = np.linalg.norm(error_field)
        
        start_invariants = self._calculate_invariants(start_grid)
        final_invariants = self._calculate_invariants(g_final)
        min_len = min(len(start_invariants), len(final_invariants))
        invariant_drift = np.sum(np.abs(start_invariants[:min_len] - final_invariants[:min_len]))
        
        unique_penalty = float(len(np.unique(g_final)))
        score = (unique_penalty * 5.0) - (error_norm * self.K_gain) - (invariant_drift * 12.0)
        return score

    def _rhea_search(self, fd, hor=6, pop=15, gen=6):
        av = []
        if hasattr(fd, 'available_actions') and fd.available_actions:
            for x in fd.available_actions:
                val = getattr(x, 'value', str(x))
                if str(val) != "RESET":
                    av.append(str(val))
        if not av:
            return "RESET"
            
        p = [[random.choice(av) for _ in range(hor)] for _ in range(pop)]
        for _ in range(gen):
            scored = sorted([(s, self._evaluate_feedback_loop(fd, s)) for s in p], key=lambda x: x[1], reverse=True)
            parents = [s for s, f in scored[:pop//2]]
            if len(parents) < 2:
                parents = parents * 2
            new_p = []
            for _ in range(pop):
                p1, p2 = random.sample(parents, 2)
                cut = random.randint(1, hor - 1)
                child = p1[:cut] + p2[cut:]
                if random.random() < 0.2:
                    child[random.randint(0, hor - 1)] = random.choice(av)
                new_p.append(child)
            p = new_p
        return scored[0][0][0]

    def choose_action(self, obs):
        self._step += 1
        
        if self.game_id in _WIN_SEQS:
            seq = _WIN_SEQS[self.game_id]
            if self._step <= len(seq):
                return GameAction.from_name(seq[self._step - 1])

        try:
            grid = self._get_grid(obs)
            sig = self._get_sig(grid)
            
            if sig == self.last_sig:
                self.sig_count += 1
            else:
                self.sig_count = 1
                self.last_sig = sig

            if self.sig_count >= 3:
                self.undo_count += 1
                self.sig_count = 0
                if self.undo_count >= 5:
                    self.undo_count = 0
                    return GameAction.from_name("RESET")
                return GameAction.from_name("ACTION7")

            av = []
            if hasattr(obs, 'available_actions') and obs.available_actions:
                for x in obs.available_actions:
                    val = getattr(x, 'value', str(x))
                    if str(val) != "RESET":
                        av.append(str(val))
            if not av:
                return GameAction.from_name("RESET")

            valid_actions = [a for a in av if _LOSS_TAB.get((self.game_id, a, sig), 0) < 3]
            if not valid_actions:
                valid_actions = av

            best_move = self._rhea_search(obs)
            if best_move not in valid_actions:
                best_move = random.choice(valid_actions)
            return GameAction.from_name(best_move)
        except Exception:
            return GameAction.from_name("RESET")


def load_game_environment(path):
    """Safely loads environment modules and handles alternative initialization patterns."""
    try:
        module_name = path.stem
        spec = importlib.util.spec_from_file_location(module_name, path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[module_name] = module
        spec.loader.exec_module(module)
        
        env = None
        if hasattr(module, 'Game'):
            env = module.Game()
        elif hasattr(module, 'Task'):
            env = module.Task()
        else:
            className = module_name.capitalize()
            if hasattr(module, className):
                env = getattr(module, className)()
                
        if env and not hasattr(env, 'reset'):
            if hasattr(env, 'start'):
                original_start = env.start
                env.reset = lambda: original_start()
            else:
                env.reset = lambda: getattr(env, 'obs', None) or getattr(env, 'state', None)
        return env
    except Exception as e:
        print(f"[WARNING] Could not load environment from {path}: {e}")
    return None


def safe_step(env, action):
    """Dynamically interfaces with the environment's step function to bypass signature errors."""
    try:
        # Standard approach: works if def step(self, action):
        return env.step(action)
    except TypeError as e:
        err_msg = str(e)
        if "positional argument" in err_msg:
            # The method exists, but argument mismatch occurred. Let's inspect it dynamically.
            sig = inspect.signature(env.step)
            params = list(sig.parameters.keys())
            
            if len(params) > 0:
                # Expects a keyword-only argument (e.g., def step(self, *, action): )
                try:
                    return env.step(**{params[0]: action})
                except Exception:
                    pass
            
            # If it takes zero arguments (e.g., def step(self): ), assign the action to state
            env.action = action
            if hasattr(env, 'current_action'):
                env.current_action = action
            return env.step()
        else:
            raise e


def main():
    all_rows = []
    game_files = list(ENV_DIR.glob("**/*.py"))
    print(f"Discovered {len(game_files)} interactive environments.")

    for g_path in game_files:
        game_id = g_path.stem
        env = load_game_environment(g_path)
        if not env:
            continue

        agent = DEQGTCrystalAgent(game_id=game_id)
        try:
            obs = env.reset() if hasattr(env, 'reset') else None
            done = False
            step_count = 0
            game_score = 0

            while not done and step_count < agent.MAX_ACTIONS:
                action = agent.choose_action(obs)
                
                # Execute dynamically inspected safe step
                step_res = safe_step(env, action)
                
                if isinstance(step_res, tuple) and len(step_res) >= 3:
                    obs, reward, done = step_res[0], step_res[1], step_res[2]
                else:
                    obs = step_res
                    reward = 1.0
                    done = getattr(env, 'is_done', lambda: False)()

                game_score += reward
                step_count += 1
                
                if hasattr(obs, 'state') and str(obs.state) == "WIN":
                    done = True

            all_rows.append({
                "row_id": f"{game_id}_0",
                "game_id": game_id,
                "end_of_game": True,
                "score": float(game_score)
            })
        except Exception as e:
            print(f"[ERROR] Execution failed on game {game_id}: {e}")
            all_rows.append({
                "row_id": f"{game_id}_0",
                "game_id": game_id,
                "end_of_game": True,
                "score": 0.0
            })

    if all_rows:
        sub_df = pd.DataFrame(all_rows)
        sub_df.to_parquet(str(PARQUET), index=False)
        print(f"Successfully generated submission parquet at {PARQUET}")

if __name__ == "__main__":
    main()

[WARNING] Kaggle secret initialization bypassed: Connection error trying to communicate with service.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


Discovered 25 interactive environments.
[ERROR] Execution failed on game sk48: property 'action' of 'Sk48' object has no setter
[ERROR] Execution failed on game tn36: property 'action' of 'Tn36' object has no setter
[ERROR] Execution failed on game m0r0: property 'action' of 'M0r0' object has no setter
[ERROR] Execution failed on game bp35: property 'action' of 'Bp35' object has no setter
[ERROR] Execution failed on game cn04: property 'action' of 'Cn04' object has no setter
[ERROR] Execution failed on game dc22: property 'action' of 'Dc22' object has no setter
[ERROR] Execution failed on game tu93: property 'action' of 'Tu93' object has no setter
[ERROR] Execution failed on game lp85: property 'action' of 'Lp85' object has no setter
[ERROR] Execution failed on game ka59: property 'action' of 'Ka59' object has no setter
[ERROR] Execution failed on game wa30: property 'action' of 'Wa30' object has no setter
[ERROR] Execution failed on game vc33: property 'action' of 'Vc33' object has no